## Setup
Environment, imports, configuration print.

In [1]:
import sys, subprocess
import os

In [2]:
import sys, os
print(sys.executable)      
print(sys.prefix)          
print(sys.version)
print(os.environ.get("VIRTUAL_ENV"), os.environ.get("CONDA_DEFAULT_ENV"))

/home/aa8624/thesis_2500/venv/bin/python
/home/aa8624/thesis_2500/venv
3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
None None


In [3]:
# List the key package versions in this environment 
print("python:", sys.executable)
print(subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
      .split("\n")[0:0] or "")
out = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
print("\n".join(l for l in out.splitlines() if l.lower().split()[0:1] and l.lower().split()[0] in
      ("torch", "torchvision", "torchaudio", "transformers", "huggingface-hub", "open_clip_torch", "open-clip-torch")))

python: /home/aa8624/thesis_2500/venv/bin/python

open_clip_torch           3.3.0
torch                     2.11.0+cu128
torchvision               0.26.0+cu128
transformers              5.13.0


In [4]:
#Pick the GPU (index from nvidia-smi)
os.environ["CUDA_VISIBLE_DEVICES"] = "3"   

In [5]:
# Imports and config printout. 
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))         
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import size_config as sc
import config
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.preprocessing import StandardScaler

%matplotlib inline
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)


print("PROJECT_ROOT :", config.PROJECT_ROOT)
print("QUICK_RUN    :", config.QUICK_RUN, "  (True = tiny settings, NOT for final results)")
print("family set   :", config.ACTIVE_FAMILY_SET, "| models:", config.MODELS_TO_RUN)
print("seeds        :", config.EXPERIMENT_SEEDS)
try:
    import torch; print("torch        :", torch.__version__, "| CUDA:", torch.cuda.is_available())
except ImportError:
    print("torch not installed (only the logistic-regression models will work)")

PROJECT_ROOT : /data/aa8624/thesis_2500
QUICK_RUN    : False   (True = tiny settings, NOT for final results)
family set   : diffusion_split | models: ['spectral_lr', 'handcrafted_lr', 'lbp_lr', 'cnn_densenet', 'supcon', 'npr', 'clip_frozen', 'biomedclip_frozen', 'rad_dino', 'dinov2_frozen']
seeds        : [42, 43, 44]
torch        : 2.11.0+cu128 | CUDA: True


In [6]:
# Number of data-loader workers for the CNN detectors
config.CNN_NUM_WORKERS = 4

In [9]:
# One-off environment fixes. Execute only once.
!{sys.executable} -m pip uninstall -y huggingface-hub
!{sys.executable} -m pip install "huggingface-hub>=1.5,<2.0"
%pip install open_clip_torch

Found existing installation: huggingface_hub 1.33.0
Uninstalling huggingface_hub-1.33.0:
  Successfully uninstalled huggingface_hub-1.33.0
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)


Note: you may need to restart the kernel to use updated packages.


## Dataset Download & Validation

In [10]:
# Show pool size, split counts and run folder
print("pool size per pool :", config.POOL_SIZE_PER_POOL)
print("split image counts :", config.SPLIT_IMAGE_COUNTS)
print("run folder         :", config.PROJECT_ROOT)

pool size per pool : 2500
split image counts : {'train': 1500, 'val': 500, 'test': 500}
run folder         : /data/aa8624/thesis_2500


In [9]:
import stage0_data.subset_from_downloaded as sub
plan = sub.build_subset("/home/aa8624/newRun_10000", drop_incomplete=True, dry_run=True, balance_splits=True)

source manifest : /home/aa8624/newRun_10000/data/manifests/real_all.csv
source images   : /home/aa8624/newRun_10000/data/real
Images listed in the source manifest but missing/empty on disk:
                      listed  missing
split disease_status                 
test  normal             928      894
      pathology         1072     1034
train normal            3117        0
      pathology         2883        0
val   normal             955        0
      pathology         1045        0
drop_incomplete: removed 681 patients with a missing image; 8011 images / 3048 patients remain
source pool     : 8011 images, 3048 patients, {'normal': 4082, 'pathology': 3929}
  424 patients contribute multiple images to this pool.

DRY RUN OK - nothing was created or copied. Plan: 2500 images, 964 patients (attempt 1)
  train: 1500 images, 558 patients, {'pathology': 750, 'normal': 750}
  val  : 500 images, 205 patients, {'pathology': 250, 'normal': 250}
  test : 500 images, 201 patients, {'normal':

In [10]:
# Copy from already downloaded path
res = sub.build_subset("/home/aa8624/newRun_10000", link="copy", drop_incomplete=True, balance_splits=True)

source manifest : /home/aa8624/newRun_10000/data/manifests/real_all.csv
source images   : /home/aa8624/newRun_10000/data/real
Images listed in the source manifest but missing/empty on disk:
                      listed  missing
split disease_status                 
test  normal             928      894
      pathology         1072     1034
train normal            3117        0
      pathology         2883        0
val   normal             955        0
      pathology         1045        0
drop_incomplete: removed 681 patients with a missing image; 8011 images / 3048 patients remain
source pool     : 8011 images, 3048 patients, {'normal': 4082, 'pathology': 3929}
  424 patients contribute multiple images to this pool.

SUBSET READY: 2500 images, 964 patients  ({'copied': 2500})
  train: 1500 images, 558 patients, {'pathology': 750, 'normal': 750}
  val  : 500 images, 205 patients, {'pathology': 250, 'normal': 250}
  test : 500 images, 201 patients, {'normal': 250, 'pathology': 250}
mani

In [11]:
import pandas as pd
from pathlib import Path
import config
from stage0_data.download_dataset import local_path

REAL, MAN = Path(config.REAL_DIR), Path(config.MANIFEST_DIR)
N = config.POOL_SIZE_PER_POOL
SPLITS, CLASSES = ("train", "val", "test"), ("normal", "pathology")
exp = {"train": N * config.TRAIN_FRACTION, "val": N * config.VAL_FRACTION, "test": N * config.TEST_FRACTION}

m = pd.read_csv(MAN / "real_all.csv", dtype={"subject_id": str, "study_id": str, "dicom_id": str})

# 1) files on disk per split x class
disk = pd.DataFrame({c: {s: len(list((REAL / s / c).glob("*.jpg"))) for s in SPLITS} for c in CLASSES})
disk["total"] = disk.sum(axis=1)
# 2) manifest rows per split x class
man = m.groupby(["split", "disease_status"]).size().unstack(fill_value=0).reindex(index=list(SPLITS), columns=list(CLASSES))
man["total"] = man.sum(axis=1)
print("FILES ON DISK\n", disk, "\n\nMANIFEST\n", man, "\n\nEXPECTED totals:", {k: int(v) for k, v in exp.items()}, "| class per split:", {k: int(v // 2) for k, v in exp.items()})

# 3) every manifest row has a file, and no extra files
expected = {local_path(REAL, r["split"], r).resolve() for _, r in m.iterrows()}
actual = {p.resolve() for p in REAL.rglob("*.jpg")}
patients = {s: set(m.loc[m.split == s, "subject_id"]) for s in SPLITS}

checks = {
    "pool size == config": len(m) == N,
    "no duplicate images": not m.duplicated(["subject_id", "study_id", "dicom_id"]).any(),
    "disk counts == manifest counts": disk.equals(man.astype(disk.dtypes)),
    "split totals as expected": all(int(man.loc[s, "total"]) == int(exp[s]) for s in SPLITS),
    "each split exactly 50/50": all(man.loc[s, "normal"] == man.loc[s, "pathology"] for s in SPLITS),
    "every manifest image exists on disk": expected <= actual,
    "no unexpected files on disk": actual <= expected,
    "no patient in two splits": all(not (patients[a] & patients[b]) for a, b in (("train", "val"), ("train", "test"), ("val", "test"))),
    "per-split CSVs match real_all": all(len(pd.read_csv(MAN / f"real_{s}.csv")) == int((m.split == s).sum()) for s in SPLITS),
}
print()
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
print("\npatients per split:", {s: len(p) for s, p in patients.items()}, "| total:", len(set.union(*patients.values())))
print("ALL CHECKS PASSED" if all(checks.values()) else "SOMETHING FAILED - see FAIL lines above")

FILES ON DISK
        normal  pathology  total
train     750        750   1500
val       250        250    500
test      250        250    500 

MANIFEST
 disease_status  normal  pathology  total
split                                   
train              750        750   1500
val                250        250    500
test               250        250    500 

EXPECTED totals: {'train': 1500, 'val': 500, 'test': 500} | class per split: {'train': 750, 'val': 250, 'test': 250}

PASS  pool size == config
PASS  no duplicate images
PASS  disk counts == manifest counts
PASS  split totals as expected
PASS  each split exactly 50/50
PASS  every manifest image exists on disk
PASS  no unexpected files on disk
PASS  no patient in two splits
PASS  per-split CSVs match real_all

patients per split: {'train': 558, 'val': 205, 'test': 201} | total: 964
ALL CHECKS PASSED


In [ ]:
# Download the real pools 
from stage0_data.download_dataset import build_and_download
build_and_download()          

MIMIC-CXR-JPG — TWO INDEPENDENT REAL IMAGE POOLS
Project root       : /data/aa8624/thesis
Pool size per pool : 500 IMAGES
Pool balance       : 250 normal / 250 pathology
Split              : 300 train / 100 validation / 100 test
Patient sampling   : MULTIPLE IMAGES PER PATIENT ALLOWED
Patient leakage    : FORBIDDEN across splits and pools
Random seed        : 42


In [12]:
# Validate the real pools; raises or prints on any problem
from stage0_data.dataset_validation import validate_dataset
validate_dataset()

DATASET VALIDATION
Pool size: 2500
Expected class counts: {'normal': 1250, 'pathology': 1250}
Expected split counts: {'train': 1500, 'val': 500, 'test': 500}

------------------------------------------------------------------------------
VALIDATING POOL: real
------------------------------------------------------------------------------

Checking manifest: /data/aa8624/thesis_2500/data/manifests/real_all.csv
Total manifest images: 2500
Class counts: {'normal': 1250, 'pathology': 1250}
Split counts: {'train': 1500, 'val': 500, 'test': 500}
Patient leakage within pool: 0
Patient-category values: PASSED
Manifest-to-file check: PASSED
JPEG/Pillow integrity check: PASSED
No unexpected JPEG files: PASSED

SUCCESS — DATASET VALIDATION PASSED
Validation report: /data/aa8624/thesis_2500/data/manifests/dataset_validation_report.txt


{'status': 'PASSED',
 'report': PosixPath('/data/aa8624/thesis_2500/data/manifests/dataset_validation_report.txt'),
 'pool_size': 2500,
 'class_counts': {'normal': 1250, 'pathology': 1250},
 'split_counts': {'train': 1500, 'val': 500, 'test': 500}}

In [1]:
import pandas as pd
from config import MANIFEST_DIR
print(pd.read_csv(MANIFEST_DIR / "real_summary.csv").to_string(index=False))
df = pd.read_csv(MANIFEST_DIR / "real_all.csv")
print("unique patients:", df["subject_id"].nunique())
print("mean images per patient:", round(len(df) / df["subject_id"].nunique(), 2))

pool split  images  unique_patients  normal_images  pathology_images
real train    1500              558            750               750
real   val     500              205            250               250
real  test     500              201            250               250
unique patients: 964
mean images per patient: 2.59
